<a href="https://colab.research.google.com/github/shin-noda/leetcode-problemset-97/blob/main/Problem913.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from collections import deque

class Solution:
    def catMouseGame(self, graph):
        # Idea: Retrograde Analysis (Bottom-Up BFS)
        # Time Complexity: O(V + E) where V = O(n^2) and E = O(n^3)
        # Space Complexity: O(n^2): to store 'color' and 'degree'

        n = len(graph)

        # State values: 0 = DRAW / UNVISITED, 1 = MOUSE_WIN, 2 = CAT_WIN
        DRAW, MOUSE_WIN, CAT_WIN = 0, 1, 2

        # colour[m][c][t] stores remaining unvisited outgoing moves for state (m, c, t)
        colour = [[[0] * 3 for _ in range(n)]  for _ in range(n)]

        # degree[m][c][t] stores remaining unvisited outgoing moves for state (m, c, t)
        degree = [[[0] * 3 for _ in range(n)] for _ in range(n)]

        # Initialize degrees for all valid states
        for m in range(n):
            for c in range(1, n):
                degree[m][c][1] = len(graph[m])
                degree[m][c][2] = sum(1 for neighbour in graph[c] if neighbour != 0)

        queue = deque()

        # 1. Initialize terminal states
        for i in range(1, n):
            for t in (1, 2):
                # Mouse reaches hole (node 0) -> Mouse Wins
                colour[0][i][t] = MOUSE_WIN
                queue.append((0, i, t, MOUSE_WIN))

                # Cat catches Mouse (same node m == c) -> Cat Wins
                colour[i][i][t] = CAT_WIN
                queue.append((i, i, t, CAT_WIN))

        # Helper to find all predecessor states of (m, c, t)
        def get_parents(m, c, t):
            parents = []

            if t == 1:
                # Current turn was Mouse's turn (t = 1)
                # Predecessor move was Cat's turn (prev_t = 2) moving to 'c'.
                for prev_c in graph[c]:
                    if prev_c != 0:
                        parents.append((m, prev_c, 2))

            else:
                # Current turn was Cat's turn (t = 2)
                # Predecessor move was Mouse's turn (prev_t = 1) moving to 'm'.
                for prev_m in graph[m]:
                    parents.append((prev_m, c, 1))

            return parents


        # 2. Bottom-up propagation
        while queue:
            m, c, t, result = queue.popleft()

            for prev_m, prev_c, prev_t in get_parents(m, c, t):
                # If parent state is already solved, skip
                if colour[prev_m][prev_c][prev_t] != DRAW:
                    continue

                # Case A: Parent player can force a win directly
                if prev_t == result:
                    colour[prev_m][prev_c][prev_t] = result
                    queue.append((prev_m, prev_c, prev_t, result))

                # Case B: Parent player moved to a losing state for them
                else:
                    degree[prev_m][prev_c][prev_t] -= 1

                    # If degree reaches 0, all possible moves lead to opponent winning
                    if degree[prev_m][prev_c][prev_t] == 0:
                        colour[prev_m][prev_c][prev_t] = result
                        queue.append((prev_m, prev_c, prev_t, result))

        return colour[1][2][1]

In [3]:
solution = Solution()

In [4]:
graph = [[2,5],[3],[0,4,5],[1,4,5],[2,3],[0,2,3]]

print(solution.catMouseGame(graph))

0


In [5]:
graph = [[1,3],[0],[3],[0,2]]

print(solution.catMouseGame(graph))

1
